<a href="https://colab.research.google.com/github/mohanasudhashanmugam/DeepLearning/blob/main/DL_Objdetection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

To download and unzip the datasets

In [1]:
!kaggle datasets download -d kipshidze/shoplifting-video-dataset


Dataset URL: https://www.kaggle.com/datasets/kipshidze/shoplifting-video-dataset
License(s): Attribution 4.0 International (CC BY 4.0)
100% 726M/726M [00:08<00:00, 86.0MB/s]



In [2]:
!unzip -q shoplifting-video-dataset.zip -d ./local_colab_storage

Dataset path Extraction



In [3]:
!ls /content/local_colab_storage/normal | wc -l

90


In [4]:
!ls /content/local_colab_storage/shoplifting | wc -l

92


In [5]:
import cv2
import os
import numpy as np
import argparse
from imutils import paths
from tensorflow.keras.utils import to_categorical
from sklearn.preprocessing import LabelBinarizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

In [6]:
# construct the argument parser and parse the arguments
ap = argparse.ArgumentParser()

ap.add_argument("-d", "--dataset",
	default="/content/local_colab_storage/",
	help="path to input dataset")
args_namespace, unknown = ap.parse_known_args()


# Convert to a dictionary
args = vars(args_namespace)


Frame extraction

In [7]:
def process_video(video_path, max_frames=32, resize_dim=(224, 224)):
    """
    Opens a video, uniformly extracts a fixed number of frames,
    resizes them, and normalizes pixel values.
    """
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    # Handle empty or corrupted videos
    if total_frames <= 0:
        cap.release()
        return None

    # Calculate uniform intervals to pick frames across the whole video duration
    # This ensures a 5-second video and a 20-second video both yield exactly 'max_frames'
    frame_indices = np.linspace(0, total_frames - 1, max_frames, dtype=int)

    video_frames = []

    for frame_idx in frame_indices:
        # Set the reader to the specific frame index
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
        success, frame = cap.read()

        if not success:
            break

        # 1. Convert color from BGR (OpenCV default) to RGB
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        # 2. Resize the frame (e.g., to 224x224)
        frame_resized = cv2.resize(frame, resize_dim)

        # 3. Normalize pixel values by dividing by 255.0 (converts 0-255 integers to 0.0-1.0 floats)
        frame_normalized = frame_resized / 255.0

        video_frames.append(frame_normalized)

    cap.release()

    # If the video didn't have enough readable frames, pad it or skip it
    if len(video_frames) < max_frames:
        return None

    # Convert list of frames into a single NumPy array
    # Output shape: (16, 224, 224, 3)
    return np.array(video_frames, dtype=np.float32)

# --- EXAMPLE USAGE ON ONE FILE ---
# (Replace with your actual unzipped path from !ls)
## sample_path = "./local_colab_storage/"


# 1. Define the video extensions you want to look for
video_extensions = (".mp4", ".avi", ".mkv", ".mov", ".wmv", ".flv", ".webm")

# 2. Grab all matching video paths recursively
video_paths = list(paths.list_files(args["dataset"], validExts=video_extensions))
print("video_paths: ", video_paths)

data=[]
labels=[]

#if os.path.exists(video_paths):
for video_path in video_paths:

  label = video_path.split(os.path.sep)[-2]
  labels.append(label)

  processed_tensor = process_video(video_path, max_frames=16, resize_dim=(224, 224))
  if processed_tensor is not None:
    data.append(processed_tensor)

  print("Video Processed Successfully!")
  print(f"Final Tensor Shape: {processed_tensor.shape}") # Expecting (16, 224, 224, 3)
  print(f"Min pixel value: {processed_tensor.min()}, Max pixel value: {processed_tensor.max()}")





video_paths:  ['/content/local_colab_storage/shoplifting/shoplifting-63.mp4', '/content/local_colab_storage/shoplifting/shoplifting-28.mp4', '/content/local_colab_storage/shoplifting/shoplifting-4.mp4', '/content/local_colab_storage/shoplifting/shoplifting-17.mp4', '/content/local_colab_storage/shoplifting/shoplifting-19.mp4', '/content/local_colab_storage/shoplifting/shoplifting-3.mp4', '/content/local_colab_storage/shoplifting/shoplifting-8.mp4', '/content/local_colab_storage/shoplifting/shoplifting-34.mp4', '/content/local_colab_storage/shoplifting/shoplifting-24.mp4', '/content/local_colab_storage/shoplifting/shoplifting-72.mp4', '/content/local_colab_storage/shoplifting/shoplifting-53.mp4', '/content/local_colab_storage/shoplifting/shoplifting-66.mp4', '/content/local_colab_storage/shoplifting/shoplifting-62.mp4', '/content/local_colab_storage/shoplifting/shoplifting-9.mp4', '/content/local_colab_storage/shoplifting/shoplifting-12.mp4', '/content/local_colab_storage/shoplifting/sh

In [8]:
##labels = np.array(labels)

In [9]:
# perform one-hot encoding on the labels
lb = LabelBinarizer()
labels = lb.fit_transform(labels)
labels = to_categorical(labels)

In [10]:
# Convert lists to final NumPy arrays
X = np.array(data, dtype=np.float32)
y = np.array(labels, dtype=np.int32)
print(f"Data loading complete!")
print(f"X shape (Videos, Frames, H, W, Channels): {X.shape}")
print(f"y shape (Labels): {y.shape}")


Data loading complete!
X shape (Videos, Frames, H, W, Channels): (182, 16, 224, 224, 3)
y shape (Labels): (182, 2)


Split into Train and Test Sets

In [11]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training samples: {X_train.shape[0]} | Testing samples: {X_test.shape[0]}")

Training samples: 145 | Testing samples: 37


CNN + LSTM

The CNN (MobileNetV2): Looks at each individual frame and extracts spatial features (like a hand, a bag, or a shelf).
The LSTM: Looks at how those spatial features change over the 16 frames to understand the action (the movement of hiding an item).

In [12]:
import tensorflow as tf
from tensorflow.keras import layers, models

# Create a sequential layer for augmentation
aug = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomContrast(0.1)
])

# 1. Base CNN to extract features from a single frame
# We use MobileNetV2 because it is lightweight and fast
base_cnn = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3), include_top=False, weights='imagenet'
)

base_cnn.trainable = False  # Freeze weights as we don't destroy pre-trained weights by new training data during backpropagation


# Flatten the CNN output to a vector
pooling_layer = layers.GlobalAveragePooling2D()(base_cnn.output)
feature_extractor = models.Model(inputs=base_cnn.input, outputs=pooling_layer)

# 2. Complete Video Model
video_input = layers.Input(shape=(32, 224, 224, 3))
# TimeDistributed applies the CNN to all frames individually
augmented_input = layers.TimeDistributed(aug)(video_input)
encoded_frames = layers.TimeDistributed(feature_extractor)(augmented_input)


# LSTM tracks the movement across the timeline
x = layers.LSTM(64, dropout=0.5)(encoded_frames)
x = layers.Dense(32, activation='relu')(x)

# Output layer: Sigmoid activation for binary classification (0 or 1)
output = layers.Dense(2, activation='sigmoid')(x)

model = models.Model(inputs=video_input, outputs=output)
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5), loss='binary_crossentropy', metrics=['accuracy'])

model.summary()


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 16, 224, 224,   │             0 │
│                                 │ 3)                     │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed                │ (None, 16, 224, 224,   │             0 │
│ (TimeDistributed)               │ 3)                     │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_1              │ (None, 16, 1280)       │     2,257,984 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 64)             │       344,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 2)              │            66 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,604,450 (9.94 MB)

 Trainable params: 346,466 (1.32 MB)

 Non-trainable params: 2,257,984 (8.61 MB)

 Training Model by feeding processed arrays into the model

In [14]:
## Run training for 30 to 50 epochs, but use an EarlyStopping callback.
#This tells Colab to keep training as long as the validation loss is improving, and automatically stop if it plateaus, saves time.

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)


history = model.fit(
    X_train, y_train,
    validation_split=0.5, # Uses a slice of training data to check performance mid-training
    epochs=30,
    batch_size=8 # Small batch size keeps memory safe
)

Epoch 1/50
9/9 ━━━━━━━━━━━━━━━━━━━━ 152s 18s/step - accuracy: 0.5139 - loss: 0.7052 - val_accuracy: 0.4932 - val_loss: 0.7037
Epoch 2/50
9/9 ━━━━━━━━━━━━━━━━━━━━ 133s 16s/step - accuracy: 0.5278 - loss: 0.6874 - val_accuracy: 0.4932 - val_loss: 0.7036
Epoch 3/50
9/9 ━━━━━━━━━━━━━━━━━━━━ 95s 11s/step - accuracy: 0.5139 - loss: 0.7063 - val_accuracy: 0.4932 - val_loss: 0.7036
Epoch 4/50
9/9 ━━━━━━━━━━━━━━━━━━━━ 134s 16s/step - accuracy: 0.5139 - loss: 0.7142 - val_accuracy: 0.4932 - val_loss: 0.7036
Epoch 5/50
9/9 ━━━━━━━━━━━━━━━━━━━━ 133s 16s/step - accuracy: 0.5139 - loss: 0.6986 - val_accuracy: 0.4932 - val_loss: 0.7035
Epoch 6/50
9/9 ━━━━━━━━━━━━━━━━━━━━ 142s 16s/step - accuracy: 0.5278 - loss: 0.6939 - val_accuracy: 0.4932 - val_loss: 0.7030
Epoch 7/50
9/9 ━━━━━━━━━━━━━━━━━━━━ 133s 16s/step - accuracy: 0.5139 - loss: 0.7143 - val_accuracy: 0.4932 - val_loss: 0.7025
Epoch 8/50
9/9 ━━━━━━━━━━━━━━━━━━━━ 140s 16s/step - accuracy: 0.5139 - loss: 0.6857 - val_accuracy: 0.4932 - val_loss: 

In [15]:
from sklearn.metrics import classification_report, confusion_matrix

# 1. Get raw probability predictions (numbers between 0.0 and 1.0)
predictions = model.predict(X_test)
print(predictions)


2/2 ━━━━━━━━━━━━━━━━━━━━ 69s 22s/step
[[0.5258877  0.49904335]
 [0.5163772  0.55454844]
 [0.5053617  0.56223536]
 [0.48131597 0.5681646 ]
 [0.4935463  0.545941  ]
 [0.50938225 0.5095782 ]
 [0.50168854 0.50709045]
 [0.52596873 0.5195574 ]
 [0.5052487  0.5375738 ]
 [0.5099579  0.55464196]
 [0.5219288  0.50604737]
 [0.49251303 0.5287794 ]
 [0.5082844  0.5910759 ]
 [0.5092244  0.5161342 ]
 [0.4912386  0.50337636]
 [0.45210308 0.53638995]
 [0.47571328 0.48773792]
 [0.48538214 0.50233805]
 [0.49064913 0.46146965]
 [0.50862914 0.5663731 ]
 [0.48472947 0.5882808 ]
 [0.51187956 0.48276985]
 [0.51739234 0.5106222 ]
 [0.4977097  0.5006082 ]
 [0.44224763 0.5504097 ]
 [0.45871487 0.5712425 ]
 [0.512201   0.5545434 ]
 [0.491055   0.5453695 ]
 [0.5101737  0.5174259 ]
 [0.5140848  0.51927274]
 [0.51065385 0.5552537 ]
 [0.54363704 0.5035585 ]
 [0.49305335 0.5225187 ]
 [0.47995856 0.5542402 ]
 [0.48943898 0.552885  ]
 [0.51628625 0.53052366]
 [0.5390789  0.5623491 ]]


In [ ]:
print(y_test)

In [2]:
# 2. Convert probabilities to binary choices: if > 0.5, it's Shoplifting (1), else Normal (0)
##binary_predictions = (predictions > 0.5).astype(int)

binary_predictions = np.argmax(predictions, axis=1)

# Convert y_test from 2 columns to 1 column
y_test_labels = np.argmax(y_test, axis=1)

print(binary_predictions)

NameError: name 'np' is not defined

In [ ]:
print("--- Confusion Matrix ---")
print(confusion_matrix(y_test_labels, binary_predictions))



In [ ]:
print("\n--- Classification Report ---")
print(classification_report(y_test_labels, binary_predictions, target_names=['Normal', 'Shoplifting']))